# MCP servers for BOOM and ML models

In this notebook you use an MCP server for BOOM in three ways:

1. Call its tools yourself, without a language model.
2. Let Claude call them, with one API request.
3. Write the agent loop yourself, which also lets you add your own model as a tool.

You need two keys in `setup/.env` (or a `.env` file next to this notebook):

- `BABAMUL_API_TOKEN`: your Babamul token from Monday (the `bbml_...` one on your profile page).
- `ANTHROPIC_API_KEY`: we'll hand this out in the session. Don't commit it or paste it anywhere shared.

In [ ]:
# Installs the session's packages if they're missing from your environment.
import importlib.util
import subprocess
import sys

if any(importlib.util.find_spec(name) is None for name in ("anthropic", "mcp", "boom_mcp")):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "anthropic>=2.6,<3", "mcp>=2.3,<3",
                           "git+https://github.com/leonitousconforti/boom-mcp"])

In [ ]:
import os
from pathlib import Path

import dotenv

for env_file in [Path(".env"), Path("../../setup/.env")]:
    if env_file.exists():
        dotenv.load_dotenv(env_file)

BABAMUL_TOKEN = os.environ["BABAMUL_API_TOKEN"]
SERVER_URL = os.environ.get("BOOM_MCP_URL", "https://boom-mcp.ltgk.net/mcp")
MODEL = "claude-opus-5-5"

print("server:", SERVER_URL)
print("Anthropic key set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## 1. Call the tools yourself

The server speaks MCP over HTTP. Each request carries your Babamul token, and the server queries BOOM as you.

In [ ]:
from contextlib import asynccontextmanager

import httpx2
from mcp.client import Client
from mcp.client.streamable_http import streamable_http_client


@asynccontextmanager
async def boom_server():
    """A connection to the hosted BOOM MCP server, using your Babamul token."""
    headers = {"Authorization": f"Bearer {BABAMUL_TOKEN}"}
    async with httpx2.AsyncClient(headers=headers, timeout=120) as http:
        async with Client(streamable_http_client(SERVER_URL, http_client=http)) as client:
            yield client

In [ ]:
async with boom_server() as boom:
    tools = (await boom.list_tools()).tools
    instructions = boom.instructions

for tool in tools:
    print(f"{tool.name:24s} {tool.description.splitlines()[0]}")

The server also sends instructions: conventions a model has to know before it can trust a result (Julian Dates, magnitudes, sentinel values, what the classifiers mean). The agent in part 3 uses them as its system prompt.

In [ ]:
print(instructions)

Now call some tools. `convert_time` with no arguments returns the current time. `search_alerts` searches BOOM; here it looks for last night's alerts that AppleCiDEr flags as out of distribution, after a few quality cuts.

In [ ]:
import json
from datetime import datetime, timedelta, timezone

async with boom_server() as boom:
    now = await boom.call_tool("convert_time", {})
print(now.content[0].text)

# Last night for ZTF (Palomar) is roughly 02:00 to 14:00 UTC.
end = datetime.now(timezone.utc).replace(hour=14, minute=0, second=0, microsecond=0)
if end > datetime.now(timezone.utc):
    end -= timedelta(days=1)
start = end - timedelta(hours=12)
print("last night:", start.isoformat(), "to", end.isoformat())

In [ ]:
async with boom_server() as boom:
    result = await boom.call_tool("search_alerts", {
        "start": start.isoformat(),
        "end": end.isoformat(),
        "min_reliability": 0.8,
        "stationary_only": True,
        "exclude_stars": True,
        "exclude_solar_system": True,
        "anomalous_only": True,
        "sort_by": "anomaly_score",
        "limit": 5,
    })

found = json.loads(result.content[0].text)
print(f"scanned {found['scanned']} alerts, {found['matched']} matched, {found['objects']} objects")
for alert in found["alerts"]:
    print(alert["object_id"], alert["time"], alert.get("mag"), alert.get("ml"))

Some tools return images. `get_cutouts` returns the science, template and difference images of an object's latest alert.

In [ ]:
import base64

from IPython.display import Image, display

object_id = found["alerts"][0]["object_id"] if found["alerts"] else "ZTF26abypkdp"

async with boom_server() as boom:
    result = await boom.call_tool("get_cutouts", {"object_id": object_id})

for block in result.content:
    if block.type == "image":
        display(Image(data=base64.b64decode(block.data)))

## 2. Let Claude call the tools

The Anthropic API can connect to an MCP server itself. You send one request with the server's URL and your Babamul token; Claude decides which tools to call, Anthropic's servers make those calls to the BOOM server, and the answer comes back with the calls listed.

In [ ]:
import anthropic

claude = anthropic.Anthropic()


def ask(question):
    """Claude answers using the BOOM server's tools through Anthropic's MCP connector."""
    messages = [{"role": "user", "content": question}]
    while True:
        response = claude.beta.messages.create(
            model=MODEL,
            max_tokens=16000,
            betas=["mcp-client-2025-11-20", "server-side-fallback-2026-07-01"],
            fallbacks="default",
            mcp_servers=[{"type": "url", "name": "boom", "url": SERVER_URL, "authorization_token": BABAMUL_TOKEN}],
            tools=[{"type": "mcp_toolset", "mcp_server_name": "boom"}],
            messages=messages,
        )
        for block in response.content:
            if block.type == "mcp_tool_use":
                print(f"-> {block.name}({json.dumps(block.input)})")
            elif block.type == "text":
                print(block.text)
        # A long run can pause partway; sending the turn back continues it.
        if response.stop_reason != "pause_turn":
            return response
        messages.append({"role": "assistant", "content": response.content})

In [ ]:
response = ask("Is ZTF26abypkdp a real transient, and what kind? Give the evidence.")

Try your own question. A few ideas:

- What did BOOM see last night that AppleCiDEr calls a TDE?
- Find objects similar to ZTF26abypkdp. Do they look alike to you?
- Which of last night's anomalous objects have a host galaxy closer than 100 Mpc?

In [ ]:
response = ask("...")

## 3. Write the agent loop yourself

Part 2 ran the agent loop inside Anthropic's API. Here it is in Python: the model asks for tool calls, your code runs them on the MCP server and sends back the results, until the model answers. Because you run the tools, this works with any MCP server, including one on your laptop with your own model added (part 5).

In [ ]:
claude_async = anthropic.AsyncAnthropic()


def claude_tool(tool):
    """An MCP tool definition in the shape the Anthropic API expects."""
    return {"name": tool.name, "description": tool.description, "input_schema": tool.input_schema}


def claude_content(result):
    """An MCP tool result (text and images) as Anthropic content blocks."""
    blocks = []
    for block in result.content:
        if block.type == "text":
            blocks.append({"type": "text", "text": block.text})
        elif block.type == "image":
            blocks.append({"type": "image",
                           "source": {"type": "base64", "media_type": block.mime_type, "data": block.data}})
    return blocks


async def agent(question, server=boom_server, show_images=False):
    """Answer a question with the tools of `server`, printing each tool call."""
    async with server() as boom:
        tools = [claude_tool(t) for t in (await boom.list_tools()).tools]
        messages = [{"role": "user", "content": question}]
        while True:
            response = await claude_async.beta.messages.create(
                model=MODEL,
                max_tokens=16000,
                system=boom.instructions or "Help the user query the BOOM alert broker.",
                tools=tools,
                messages=messages,
                cache_control={"type": "ephemeral"},
                betas=["server-side-fallback-2026-07-01"],
                fallbacks="default",
            )
            messages.append({"role": "assistant", "content": response.content})
            calls = [block for block in response.content if block.type == "tool_use"]
            if not calls:
                break
            results = []
            for call in calls:
                print(f"-> {call.name}({json.dumps(call.input)})")
                result = await boom.call_tool(call.name, call.input)
                if show_images:
                    for block in result.content:
                        if block.type == "image":
                            display(Image(data=base64.b64decode(block.data)))
                results.append({"type": "tool_result", "tool_use_id": call.id,
                                "content": claude_content(result), "is_error": result.is_error})
            messages.append({"role": "user", "content": results})
    print("".join(block.text for block in response.content if block.type == "text"))
    return messages

In [ ]:
messages = await agent("Pick one of last night's anomalous ZTF objects with a host galaxy and tell me why it was flagged.",
                       show_images=True)

## 4. Activities

**Warm-up.** Ask your agent the question below, then compare with your neighbor. Did you get the same objects? Which cuts did your agent choose?

**The hunt.** The start card is on the screen. Use `ask(...)` or `agent(...)`, whichever you like.

**Bingo.** The squares are on the screen. When you fill one, keep the prompt that caused it.

In [ ]:
messages = await agent("What were the three most interesting new transients last night, and why?")

## 5. Add your own model

`my_model.py` in this folder is a plugin for the server: every function in it marked `@tool()` becomes a tool. It comes with one working tool, `color_change`, and a sketch for the LUNA pipeline from this morning.

`local_server` runs the BOOM MCP server on your laptop with your plugin loaded, and `agent` can use it in place of the hosted server. It still queries BOOM with your Babamul token.

In [ ]:
from mcp.client.stdio import StdioServerParameters


def local_server(*plugins):
    """The BOOM MCP server running on your laptop, with your plugin files loaded."""
    args = ["-m", "boom_mcp.server", *[arg for plugin in plugins for arg in ("--plugin", plugin)]]
    return Client(StdioServerParameters(command=sys.executable, args=args, env=dict(os.environ)))


async with local_server("my_model.py") as boom:
    result = await boom.call_tool("color_change", {"object_id": "ZTF26abypkdp"})
print(result.content[0].text)

In [ ]:
messages = await agent("Has ZTF26abypkdp gotten redder since it was discovered? What does that say about it?",
                       server=lambda: local_server("my_model.py"))

Now make it yours: replace `color_change` with your own model, or add a second tool next to it. Each call to `agent` starts a fresh server, so your edits take effect on the next run.

Write the docstring for the model, not for a person. Say what the output means (is a high score good or bad?), or the agent will guess.

## More

- Server code: [github.com/leonitousconforti/boom-mcp](https://github.com/leonitousconforti/boom-mcp)
- SkyPortal's MCP endpoint: [skyportal/skyportal, doc/mcp.md](https://github.com/skyportal/skyportal/blob/main/doc/mcp.md)
- The protocol: [modelcontextprotocol.io](https://modelcontextprotocol.io)